## 1. Setup and Imports

In [ ]:
import sys
import os

# Add parent directory to path
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import linalg, optimize, stats, signal
from scipy.fft import fft, fftfreq, ifft
import warnings
warnings.filterwarnings('ignore')

# Plotting configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("  Imports successful")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 2. Linear Algebra Operations

Foundation for portfolio optimization, factor models, and dimensionality reduction.

In [ ]:
# Generate sample covariance matrix for portfolio analysis
n_assets = 10
np.random.seed(42)

# Create random correlation matrix
random_matrix = np.random.randn(n_assets, n_assets)
correlation_matrix = random_matrix @ random_matrix.T

# Add noise for numerical stability
correlation_matrix = correlation_matrix + np.eye(n_assets) * 0.1

# Normalize to correlation matrix
D = np.diag(1 / np.sqrt(np.diag(correlation_matrix)))
correlation_matrix = D @ correlation_matrix @ D

print("Correlation Matrix Shape:", correlation_matrix.shape)
print("\nMatrix Properties:")
print(f"  - Symmetric: {np.allclose(correlation_matrix, correlation_matrix.T)}")
print(f"  - Positive Definite: {np.all(np.linalg.eigvals(correlation_matrix) > 0)}")

In [ ]:
# Eigenvalue decomposition for Principal Component Analysis
eigenvalues, eigenvectors = linalg.eigh(correlation_matrix)

# Sort in descending order
idx = eigenvalues.argsort()[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# Calculate explained variance ratio
explained_variance_ratio = eigenvalues / eigenvalues.sum()
cumulative_variance = np.cumsum(explained_variance_ratio)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].bar(range(1, n_assets + 1), explained_variance_ratio, alpha=0.7, label='Individual')
axes[0].plot(range(1, n_assets + 1), cumulative_variance, 'ro-', label='Cumulative')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Eigenvalue Decomposition - Scree Plot')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Correlation heatmap
im = axes[1].imshow(correlation_matrix, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
axes[1].set_title('Asset Correlation Matrix')
axes[1].set_xlabel('Asset Index')
axes[1].set_ylabel('Asset Index')
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

print(f"\nFirst 3 components explain {cumulative_variance[2]*100:.2f}% of variance")
print(f"First 5 components explain {cumulative_variance[4]*100:.2f}% of variance")

In [ ]:
# Cholesky decomposition for Monte Carlo simulation
L = linalg.cholesky(correlation_matrix, lower=True)

# Verify: L @ L.T = Correlation Matrix
reconstructed = L @ L.T
reconstruction_error = np.max(np.abs(reconstructed - correlation_matrix))

print("Cholesky Decomposition:")
print(f"  - Matrix shape: {L.shape}")
print(f"  - Lower triangular: {np.allclose(L, np.tril(L))}")
print(f"  - Reconstruction error: {reconstruction_error:.2e}")

# Generate correlated random samples
n_samples = 1000
independent_samples = np.random.randn(n_samples, n_assets)
correlated_samples = independent_samples @ L.T

# Verify correlation structure
sample_correlation = np.corrcoef(correlated_samples.T)
correlation_error = np.max(np.abs(sample_correlation - correlation_matrix))

print(f"\nMonte Carlo Simulation:")
print(f"  - Generated {n_samples} correlated samples")
print(f"  - Sample correlation error: {correlation_error:.4f}")

## 3. Stochastic Calculus

Essential for option pricing, risk management, and market microstructure modeling.

In [ ]:
# Geometric Brownian Motion (GBM) simulation
def simulate_gbm(S0, mu, sigma, T, dt, n_paths):
    """
    Simulate Geometric Brownian Motion paths
    dS = μS dt + σS dW
    """
    n_steps = int(T / dt)
    t = np.linspace(0, T, n_steps)
    
    # Generate Brownian motion increments
    dW = np.random.randn(n_paths, n_steps) * np.sqrt(dt)
    
    # Calculate cumulative returns
    cumulative_returns = (mu - 0.5 * sigma**2) * dt + sigma * dW
    cumulative_returns[:, 0] = 0  # Start at zero
    cumulative_returns = np.cumsum(cumulative_returns, axis=1)
    
    # Calculate price paths
    S = S0 * np.exp(cumulative_returns)
    
    return t, S

# Simulation parameters
S0 = 100  # Initial price
mu = 0.15  # Drift (15% annual return)
sigma = 0.25  # Volatility (25% annual vol)
T = 1.0  # Time horizon (1 year)
dt = 1/252  # Time step (daily)
n_paths = 5

t, S = simulate_gbm(S0, mu, sigma, T, dt, n_paths)

# Visualization
plt.figure(figsize=(12, 6))
for i in range(n_paths):
    plt.plot(t, S[i, :], alpha=0.7, label=f'Path {i+1}')

plt.axhline(y=S0, color='black', linestyle='--', alpha=0.5, label='Initial Price')
plt.xlabel('Time (years)')
plt.ylabel('Price ($)')
plt.title('Geometric Brownian Motion - Stock Price Simulation')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Initial Price: ${S0:.2f}")
print(f"Final Prices: ${S[:, -1].min():.2f} - ${S[:, -1].max():.2f}")
print(f"Average Final Price: ${S[:, -1].mean():.2f}")

In [ ]:
# Ornstein-Uhlenbeck process for mean reversion
def simulate_ou_process(x0, theta, mu, sigma, T, dt, n_paths):
    """
    Simulate Ornstein-Uhlenbeck process
    dx = θ(μ - x) dt + σ dW
    """
    n_steps = int(T / dt)
    t = np.linspace(0, T, n_steps)
    
    x = np.zeros((n_paths, n_steps))
    x[:, 0] = x0
    
    for i in range(1, n_steps):
        dW = np.random.randn(n_paths) * np.sqrt(dt)
        x[:, i] = x[:, i-1] + theta * (mu - x[:, i-1]) * dt + sigma * dW
    
    return t, x

# Mean reversion parameters
x0 = 0.0  # Initial value
theta = 3.0  # Mean reversion speed
mu = 0.0  # Long-term mean
sigma = 0.5  # Volatility
T = 2.0  # Time horizon
dt = 0.01  # Time step
n_paths = 100

t, x = simulate_ou_process(x0, theta, mu, sigma, T, dt, n_paths)

# Calculate theoretical properties
half_life = np.log(2) / theta
stationary_std = sigma / np.sqrt(2 * theta)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time series
axes[0].plot(t, x[:5, :].T, alpha=0.6)
axes[0].axhline(y=mu, color='red', linestyle='--', label=f'Mean = {mu}')
axes[0].fill_between(t, mu - stationary_std, mu + stationary_std, 
                      alpha=0.2, color='gray', label='±1 Std Dev')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Value')
axes[0].set_title('Ornstein-Uhlenbeck Process - Mean Reversion')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Distribution at final time
axes[1].hist(x[:, -1], bins=30, density=True, alpha=0.7, edgecolor='black')
axes[1].axvline(x=mu, color='red', linestyle='--', label='Mean')
axes[1].set_xlabel('Final Value')
axes[1].set_ylabel('Probability Density')
axes[1].set_title('Distribution at T = 2.0')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean Reversion Properties:")
print(f"  - Half-life: {half_life:.4f}")
print(f"  - Stationary std dev: {stationary_std:.4f}")
print(f"  - Empirical final std dev: {x[:, -1].std():.4f}")

## 4. Fourier Analysis

Signal processing for market cycles, periodicity detection, and filtering.

In [ ]:
# Generate synthetic signal with multiple frequencies
sampling_rate = 1000  # Hz
duration = 2.0  # seconds
t = np.linspace(0, duration, int(sampling_rate * duration), endpoint=False)

# Create signal with multiple frequency components
signal_1 = 2.0 * np.sin(2 * np.pi * 50 * t)  # 50 Hz
signal_2 = 1.5 * np.sin(2 * np.pi * 120 * t)  # 120 Hz
signal_3 = 0.5 * np.sin(2 * np.pi * 300 * t)  # 300 Hz
noise = 0.3 * np.random.randn(len(t))

composite_signal = signal_1 + signal_2 + signal_3 + noise

# Compute FFT
fft_values = fft(composite_signal)
fft_freq = fftfreq(len(composite_signal), 1/sampling_rate)

# Only positive frequencies
positive_freq_idx = fft_freq > 0
frequencies = fft_freq[positive_freq_idx]
magnitude = np.abs(fft_values[positive_freq_idx])
power = magnitude ** 2

# Visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Time domain
axes[0].plot(t[:500], composite_signal[:500], 'b-', alpha=0.7, linewidth=0.5)
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Time Domain Signal')
axes[0].grid(True, alpha=0.3)

# Frequency domain
axes[1].plot(frequencies, power, 'r-', linewidth=1)
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Power')
axes[1].set_title('Frequency Domain (Power Spectrum)')
axes[1].set_xlim(0, 400)
axes[1].grid(True, alpha=0.3)

# Mark dominant frequencies
peaks_idx = np.argsort(power)[-3:]  # Top 3 peaks
for idx in peaks_idx:
    axes[1].axvline(x=frequencies[idx], color='green', linestyle='--', alpha=0.5)
    axes[1].text(frequencies[idx], power[idx], f'{frequencies[idx]:.0f} Hz', 
                 rotation=90, va='bottom')

plt.tight_layout()
plt.show()

print("Detected Dominant Frequencies (Hz):")
for idx in peaks_idx:
    print(f"  - {frequencies[idx]:.2f} Hz (Power: {power[idx]:.2e})")

In [ ]:
# Band-pass filter using FFT
def bandpass_filter(signal, lowcut, highcut, sampling_rate):
    """
    Apply band-pass filter in frequency domain
    """
    fft_signal = fft(signal)
    freq = fftfreq(len(signal), 1/sampling_rate)
    
    # Create filter mask
    mask = (np.abs(freq) >= lowcut) & (np.abs(freq) <= highcut)
    
    # Apply filter
    fft_filtered = fft_signal * mask
    
    # Inverse FFT
    filtered_signal = np.real(ifft(fft_filtered))
    
    return filtered_signal

# Apply filter to extract 120 Hz component
filtered_signal = bandpass_filter(composite_signal, lowcut=100, highcut=140, 
                                  sampling_rate=sampling_rate)

# Visualization
plt.figure(figsize=(14, 6))
plt.plot(t[:500], composite_signal[:500], 'b-', alpha=0.5, linewidth=0.8, label='Original')
plt.plot(t[:500], filtered_signal[:500], 'r-', linewidth=1.5, label='Filtered (100-140 Hz)')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.title('Band-Pass Filter Application')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Original signal std: {composite_signal.std():.4f}")
print(f"Filtered signal std: {filtered_signal.std():.4f}")
print(f"Noise reduction: {(1 - filtered_signal.std()/composite_signal.std())*100:.2f}%")

## 5. Numerical Methods

Optimization and root-finding for calibration and parameter estimation.

In [ ]:
# Portfolio optimization using numerical methods
def portfolio_variance(weights, cov_matrix):
    """Calculate portfolio variance"""
    return weights.T @ cov_matrix @ weights

def portfolio_return(weights, expected_returns):
    """Calculate portfolio expected return"""
    return weights.T @ expected_returns

# Generate sample data
n_assets = 5
np.random.seed(42)
expected_returns = np.random.uniform(0.05, 0.20, n_assets)
volatilities = np.random.uniform(0.10, 0.30, n_assets)

# Create covariance matrix
random_corr = np.random.uniform(-0.3, 0.7, (n_assets, n_assets))
correlation = (random_corr + random_corr.T) / 2
np.fill_diagonal(correlation, 1.0)

# Ensure positive definite
eigenvals, eigenvecs = np.linalg.eigh(correlation)
eigenvals = np.maximum(eigenvals, 0.01)
correlation = eigenvecs @ np.diag(eigenvals) @ eigenvecs.T

# Normalize to correlation matrix
D = np.diag(1 / np.sqrt(np.diag(correlation)))
correlation = D @ correlation @ D

# Create covariance matrix
cov_matrix = np.diag(volatilities) @ correlation @ np.diag(volatilities)

print("Portfolio Optimization Setup:")
print(f"  - Number of assets: {n_assets}")
print(f"  - Expected returns: {expected_returns}")
print(f"  - Volatilities: {volatilities}")

In [ ]:
# Minimum variance portfolio
def minimize_variance(cov_matrix):
    n = len(cov_matrix)
    
    # Objective: minimize portfolio variance
    objective = lambda w: portfolio_variance(w, cov_matrix)
    
    # Constraints: weights sum to 1
    constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
    
    # Bounds: no short selling
    bounds = tuple((0, 1) for _ in range(n))
    
    # Initial guess: equal weights
    x0 = np.ones(n) / n
    
    # Optimize
    result = optimize.minimize(objective, x0, method='SLSQP', 
                              bounds=bounds, constraints=constraints)
    
    return result

# Maximum Sharpe ratio portfolio (risk-free rate = 2%)
def maximize_sharpe(expected_returns, cov_matrix, risk_free_rate=0.02):
    n = len(expected_returns)
    
    # Objective: maximize Sharpe ratio (minimize negative Sharpe)
    def negative_sharpe(w):
        portfolio_ret = portfolio_return(w, expected_returns)
        portfolio_vol = np.sqrt(portfolio_variance(w, cov_matrix))
        return -(portfolio_ret - risk_free_rate) / portfolio_vol
    
    constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
    bounds = tuple((0, 1) for _ in range(n))
    x0 = np.ones(n) / n
    
    result = optimize.minimize(negative_sharpe, x0, method='SLSQP',
                              bounds=bounds, constraints=constraints)
    
    return result

# Optimize portfolios
min_var_result = minimize_variance(cov_matrix)
max_sharpe_result = maximize_sharpe(expected_returns, cov_matrix)

# Calculate metrics
min_var_weights = min_var_result.x
min_var_return = portfolio_return(min_var_weights, expected_returns)
min_var_vol = np.sqrt(portfolio_variance(min_var_weights, cov_matrix))

max_sharpe_weights = max_sharpe_result.x
max_sharpe_return = portfolio_return(max_sharpe_weights, expected_returns)
max_sharpe_vol = np.sqrt(portfolio_variance(max_sharpe_weights, cov_matrix))
max_sharpe_ratio = (max_sharpe_return - 0.02) / max_sharpe_vol

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Weights comparison
x = np.arange(n_assets)
width = 0.35
axes[0].bar(x - width/2, min_var_weights, width, label='Min Variance', alpha=0.8)
axes[0].bar(x + width/2, max_sharpe_weights, width, label='Max Sharpe', alpha=0.8)
axes[0].set_xlabel('Asset Index')
axes[0].set_ylabel('Weight')
axes[0].set_title('Portfolio Weights Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Efficient frontier
n_portfolios = 100
returns_range = np.linspace(expected_returns.min(), expected_returns.max(), n_portfolios)
efficient_vols = []

for target_return in returns_range:
    constraints = [
        {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
        {'type': 'eq', 'fun': lambda w: portfolio_return(w, expected_returns) - target_return}
    ]
    
    result = optimize.minimize(
        lambda w: portfolio_variance(w, cov_matrix),
        np.ones(n_assets) / n_assets,
        method='SLSQP',
        bounds=tuple((0, 1) for _ in range(n_assets)),
        constraints=constraints
    )
    
    if result.success:
        efficient_vols.append(np.sqrt(result.fun))
    else:
        efficient_vols.append(np.nan)

axes[1].plot(efficient_vols, returns_range, 'b-', linewidth=2, label='Efficient Frontier')
axes[1].scatter(min_var_vol, min_var_return, c='green', s=100, marker='*', 
                label='Min Variance', zorder=5)
axes[1].scatter(max_sharpe_vol, max_sharpe_return, c='red', s=100, marker='*', 
                label='Max Sharpe', zorder=5)
axes[1].set_xlabel('Volatility (Risk)')
axes[1].set_ylabel('Expected Return')
axes[1].set_title('Efficient Frontier')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nOptimization Results:")
print("\nMinimum Variance Portfolio:")
print(f"  - Expected Return: {min_var_return*100:.2f}%")
print(f"  - Volatility: {min_var_vol*100:.2f}%")
print(f"  - Sharpe Ratio: {(min_var_return - 0.02)/min_var_vol:.4f}")

print("\nMaximum Sharpe Ratio Portfolio:")
print(f"  - Expected Return: {max_sharpe_return*100:.2f}%")
print(f"  - Volatility: {max_sharpe_vol*100:.2f}%")
print(f"  - Sharpe Ratio: {max_sharpe_ratio:.4f}")

## 6. Statistical Tests

Hypothesis testing for strategy validation and risk management.

In [ ]:
# Generate sample returns data
np.random.seed(42)
n_days = 252

# Strategy returns (with positive drift)
strategy_returns = np.random.normal(0.0005, 0.015, n_days)  # 0.05% daily mean

# Benchmark returns
benchmark_returns = np.random.normal(0.0003, 0.012, n_days)  # 0.03% daily mean

# Calculate cumulative returns
strategy_cumulative = np.cumprod(1 + strategy_returns) - 1
benchmark_cumulative = np.cumprod(1 + benchmark_returns) - 1

# Visualization
plt.figure(figsize=(14, 6))
plt.plot(strategy_cumulative * 100, label='Strategy', linewidth=2)
plt.plot(benchmark_cumulative * 100, label='Benchmark', linewidth=2, alpha=0.7)
plt.xlabel('Trading Days')
plt.ylabel('Cumulative Return (%)')
plt.title('Strategy vs Benchmark Performance')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Strategy final return: {strategy_cumulative[-1]*100:.2f}%")
print(f"Benchmark final return: {benchmark_cumulative[-1]*100:.2f}%")

In [ ]:
# Statistical tests

# 1. Normality test (Jarque-Bera)
jb_stat, jb_pvalue = stats.jarque_bera(strategy_returns)

# 2. T-test for mean > 0
t_stat, t_pvalue = stats.ttest_1samp(strategy_returns, 0)

# 3. Paired t-test (strategy vs benchmark)
paired_t_stat, paired_t_pvalue = stats.ttest_rel(strategy_returns, benchmark_returns)

# 4. Kolmogorov-Smirnov test
ks_stat, ks_pvalue = stats.kstest(strategy_returns, 'norm', 
                                   args=(strategy_returns.mean(), strategy_returns.std()))

# 5. Ljung-Box test for autocorrelation
from statsmodels.stats.diagnostic import acorr_ljungbox
lb_result = acorr_ljungbox(strategy_returns, lags=10, return_df=True)

print("Statistical Test Results:")
print("="*60)

print("\n1. Jarque-Bera Normality Test:")
print(f"   Statistic: {jb_stat:.4f}")
print(f"   P-value: {jb_pvalue:.4f}")
print(f"   Result: {'Normal' if jb_pvalue > 0.05 else 'Not Normal'} at 5% level")

print("\n2. One-Sample T-Test (Mean > 0):")
print(f"   Statistic: {t_stat:.4f}")
print(f"   P-value: {t_pvalue/2:.4f}")
print(f"   Result: {'Reject H0' if t_pvalue/2 < 0.05 else 'Fail to reject H0'} at 5% level")

print("\n3. Paired T-Test (Strategy vs Benchmark):")
print(f"   Statistic: {paired_t_stat:.4f}")
print(f"   P-value: {paired_t_pvalue:.4f}")
print(f"   Result: Strategy {'outperforms' if paired_t_pvalue < 0.05 else 'similar to'} benchmark")

print("\n4. Kolmogorov-Smirnov Test:")
print(f"   Statistic: {ks_stat:.4f}")
print(f"   P-value: {ks_pvalue:.4f}")

print("\n5. Ljung-Box Test (Autocorrelation):")
print(f"   Lag 1 p-value: {lb_result.iloc[0]['lb_pvalue']:.4f}")
print(f"   Lag 5 p-value: {lb_result.iloc[4]['lb_pvalue']:.4f}")
print(f"   Lag 10 p-value: {lb_result.iloc[9]['lb_pvalue']:.4f}")

In [ ]:
# Distribution analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram
axes[0, 0].hist(strategy_returns, bins=30, density=True, alpha=0.7, edgecolor='black')
axes[0, 0].axvline(x=strategy_returns.mean(), color='red', linestyle='--', 
                   label=f'Mean = {strategy_returns.mean()*100:.3f}%')
axes[0, 0].set_xlabel('Daily Returns')
axes[0, 0].set_ylabel('Density')
axes[0, 0].set_title('Returns Distribution')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Q-Q plot
stats.probplot(strategy_returns, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot (Normality Check)')
axes[0, 1].grid(True, alpha=0.3)

# ACF plot
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(strategy_returns, lags=20, ax=axes[1, 0])
axes[1, 0].set_title('Autocorrelation Function')
axes[1, 0].grid(True, alpha=0.3)

# Rolling statistics
rolling_window = 20
rolling_mean = pd.Series(strategy_returns).rolling(rolling_window).mean()
rolling_std = pd.Series(strategy_returns).rolling(rolling_window).std()

axes[1, 1].plot(strategy_returns, alpha=0.5, label='Returns')
axes[1, 1].plot(rolling_mean, 'r-', label=f'{rolling_window}-day Mean')
axes[1, 1].fill_between(range(len(rolling_std)), 
                        rolling_mean - 2*rolling_std, 
                        rolling_mean + 2*rolling_std, 
                        alpha=0.2, label='±2 Std')
axes[1, 1].set_xlabel('Days')
axes[1, 1].set_ylabel('Returns')
axes[1, 1].set_title('Rolling Statistics')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics
print("\nSummary Statistics:")
print(f"  - Mean: {strategy_returns.mean()*100:.4f}%")
print(f"  - Median: {np.median(strategy_returns)*100:.4f}%")
print(f"  - Std Dev: {strategy_returns.std()*100:.4f}%")
print(f"  - Skewness: {stats.skew(strategy_returns):.4f}")
print(f"  - Kurtosis: {stats.kurtosis(strategy_returns):.4f}")
print(f"  - Sharpe Ratio: {(strategy_returns.mean() / strategy_returns.std()) * np.sqrt(252):.4f}")

## 7. Summary and Key Takeaways

This notebook demonstrated the mathematical foundations essential for quantitative trading:

### Key Components:

1. **Linear Algebra**
   - Eigenvalue decomposition for PCA and risk factor analysis
   - Cholesky decomposition for Monte Carlo simulations
   - Applications in portfolio optimization and covariance estimation

2. **Stochastic Calculus**
   - Geometric Brownian Motion for price modeling
   - Ornstein-Uhlenbeck process for mean reversion strategies
   - Foundation for derivatives pricing and risk-neutral valuation

3. **Fourier Analysis**
   - Frequency domain analysis for cycle detection
   - Signal filtering and noise reduction
   - Spectral analysis for market microstructure

4. **Numerical Optimization**
   - Portfolio optimization (minimum variance, maximum Sharpe)
   - Efficient frontier construction
   - Constrained optimization for real-world constraints

5. **Statistical Testing**
   - Hypothesis testing for strategy validation
   - Normality and distribution tests
   - Autocorrelation analysis

### Next Steps:
- **Signal Processing**: Advanced filtering and feature extraction
- **Machine Learning**: Model training and evaluation
- **Backtesting**: Strategy validation and performance analysis